In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import numpy as np
import torch
from ipywidgets import interact

print(torch.version.cuda)
print(torch.cuda.is_available())

This alternative formulation allows you to immediately compute the smoothing at a given step without first generating all steps before it.
This is easier to work wit and is closer to how normal diffusino models work.
It means we can generate a random amount of noise on the go.

However, getting it to work properly is difficult.

Problem is that it relies on absolute distances.

In [ ]:

def directed_random_walk_diffusion_expm(
    x: torch.Tensor,  # shape (N,) or (N,C)
    d: torch.Tensor,  # shape (N,N), directed distances d[i,j]
    s: float,         # diffusion time (0 < s < 1)
    sigma: float,     # kernel scale (> 0)
) -> torch.Tensor:
    """One-step directed random-walk diffusion smoothing via matrix exponential.

    Defines:
      K_ij = exp(-d_ij / sigma)
      P_ij = K_ij / sum_j K_ij           (row-stochastic)
      L    = I - P                       (random-walk Laplacian / generator)
      x'   = exp(-t * L) @ x             (closed-form diffusion at time t)

    Args:
      x: Values at N samples. Shape (N,) or (N,C).
      d: Directed distance matrix. Shape (N,N).
      s: Diffusion time/strength (0 < s < 1).
      sigma: Kernel scale.

    Returns:
      Smoothed x with the same shape as input.
    """
    t = s / (1 - s)  # Reparameterize s in (0,1) to t in (0,inf)
    K = torch.exp(-d / sigma)                           # (N,N)
    P = K / K.sum(dim=1, keepdim=True)                  # (N,N)
    N = d.shape[0]
    L = torch.eye(N, device=d.device, dtype=d.dtype) - P  # (N,N)
    return torch.linalg.matrix_exp(-t * L) @ x          # (N,) or (N,C)

In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

torch.manual_seed(0)
data = torch.rand(250, 2, device='cuda')

@interact(time=(0.01, 0.99, 0.01), log_sigma=(-5, -1, 0.01), scale=(0.5, 2.0, 0.01), offset=(-5, 5, 0.01),)
def show(time, log_sigma, scale, offset):
    sigma = np.exp(log_sigma)
    ax.clear()
    ax.scatter(data[:, 0].cpu().numpy(), data[:, 1].cpu().numpy(), marker='o', s=10, color='gray', alpha=0.25)
    dists = torch.linalg.vector_norm(data[:, None, :] - data[None, :, :], dim=-1)  # (N,N)
    # dists = dists ** 2
    # dists = 1/ dists
    # dists = scale * dists + offset
    # dists = 1 / (1 + torch.exp(- dists))
    dists = torch.exp(dists * scale + offset)
    data_transformed = directed_random_walk_diffusion_expm(
        x=data,
        d=dists,
        s=time,
        sigma=sigma
    )
    ax.scatter(data_transformed[:, 0].cpu().numpy(), data_transformed[:, 1].cpu().numpy(), marker='o', s=10)
    ax.set_title(f"time={time:.2f}, sigma={sigma:.4f}")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)